In [1]:
import os
import sys
import time, math
from itertools import product

from libs.multilevel_squeme import (
    multilevel_bipartition,
    edge_cut,
    k_way_partition,
    edge_cut_kway,
    kway_balance_info,
    partition_graph_metis,
)

from libs.utils import (
    load_mtx,
    matrix_to_graph,
    summarize_generic
)

In [2]:
# Diagnostics: interpreter and METIS backend
print("Python executable:", sys.executable)

try:
    import pymetis
    print("PyMetis:", getattr(pymetis, "__file__", "?"))
except Exception as e:
    print("PyMetis import failed:", repr(e))

Python executable: /home/operation/reps/FEM-Graph-Partitioning-Toolkit-Multilevel-METIS-/.venv/bin/python
PyMetis: /home/operation/reps/FEM-Graph-Partitioning-Toolkit-Multilevel-METIS-/.venv/lib/python3.10/site-packages/pymetis/__init__.py


In [3]:
data_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "data"))

k_matrix_path = os.path.join(data_dir, "hybrid_ma.classical.fem.matrix_k.mtx")
m_matrix_path = os.path.join(data_dir, "hybrid_ma.classical.fem.matrix_m.mtx")
print("Using data_dir:", data_dir)

Using data_dir: /home/operation/reps/FEM-Graph-Partitioning-Toolkit-Multilevel-METIS-/data


In [4]:
# Load matrices
A_K = load_mtx(k_matrix_path, 'K')
A_M = load_mtx(m_matrix_path, 'M')

if A_K is None or A_M is None:
    raise RuntimeError("Matrix load failed; check file paths printed above.")

print(f"Loaded: K | shape={A_K.shape}, nnz={A_K.nnz}")
print(f"Loaded: M | shape={A_M.shape}, nnz={A_M.nnz}")

Loaded: K | shape=(960, 960), nnz=30282
Loaded: M | shape=(960, 960), nnz=10098
Loaded: K | shape=(960, 960), nnz=30282
Loaded: M | shape=(960, 960), nnz=10098


In [5]:
# Build graphs from matrices
diag_K = A_K.diagonal()
diag_M = A_M.diagonal()

G_K = matrix_to_graph(A_K, symmetrize='sum', drop_diagonal=True, abs_weights=True, node_vweight='diag', diag=diag_K)
G_M = matrix_to_graph(A_M, symmetrize='sum', drop_diagonal=True, abs_weights=True, node_vweight='diag', diag=diag_M)

print(f"K: |V|={G_K.number_of_nodes()}, |E|={G_K.number_of_edges()}")
print(f"M: |V|={G_M.number_of_nodes()}, |E|={G_M.number_of_edges()} (vweights set from diagonal)")

K: |V|=960, |E|=14661
M: |V|=960, |E|=4569 (vweights set from diagonal)


In [6]:
# Partitioning configuration
# Modes: 'bipartition' | 'kway_recursive' | 'kway_direct' | 'kway_metis'
PARTITION_MODE = 'kway_direct'
NPARTS = 5  # keep 2 for classic bipartition

In [7]:
# Run partitions based on PARTITION_MODE and NPARTS
# Default params per graph
params_K = dict(weight='weight', initial_method='GGGP', refine_method='FM',
                balance_tol=0.03, max_levels=35, coarsen_limit=40, refine_passes=24, n_trials=8, seed=42)
params_M = dict(weight='weight', initial_method='COMPONENT_AWARE', refine_method='FM',
                balance_tol=0.06, max_levels=35, coarsen_limit=60, refine_passes=20, n_trials=10, seed=123)

if PARTITION_MODE == 'bipartition' and NPARTS == 2:
    part_K = multilevel_bipartition(G_K, **params_K)
    part_M = multilevel_bipartition(G_M, **params_M)
elif PARTITION_MODE == 'kway_recursive':
    part_K = k_way_partition(G_K, NPARTS, **params_K)
    part_M = k_way_partition(G_M, NPARTS, **params_M)
elif PARTITION_MODE == 'kway_direct':
    # Direct K-way multilevel with K-way refinement and final rebalance
    from libs.multilevel_squeme import multilevel_kway_partition
    dparams_K = dict(params_K)
    dparams_M = dict(params_M)
    dparams_K.setdefault('refine_method', 'KFM')
    dparams_M.setdefault('refine_method', 'KFM')
    part_K = multilevel_kway_partition(G_K, NPARTS, **dparams_K)
    part_M = multilevel_kway_partition(G_M, NPARTS, **dparams_M)
elif PARTITION_MODE == 'kway_metis':
    part_K = partition_graph_metis(G_K, nparts=NPARTS, weight='weight', seed=params_K.get('seed', 42))
    part_M = partition_graph_metis(G_M, nparts=NPARTS, weight='weight', seed=params_M.get('seed', 123))
else:
    raise ValueError(f"Unsupported configuration: PARTITION_MODE={PARTITION_MODE}, NPARTS={NPARTS}")

summarize_generic(G_K, part_K, f"K [{PARTITION_MODE} {NPARTS}]")
summarize_generic(G_M, part_M, f"M [{PARTITION_MODE} {NPARTS}]")

K [kway_direct 5]: cut=10588.7117, parts=[0, 1, 2, 3, 4], per=[6558.285588286818, 6337.232554128872, 6912.812984045712, 6922.310719176806, 6915.223128203641], total=33645.8650
M [kway_direct 5]: cut=11202.2331, parts=[0, 1, 2, 3, 4], per=[30515.922496431947, 29891.705039255878, 48486.104602346415, 37129.46218391173, 35199.688285117496], total=181222.8826


In [8]:
# Baseline METIS comparison for the same NPARTS
try:
    metis_part_K = partition_graph_metis(G_K, nparts=NPARTS, weight='weight', seed=42)
    summarize_generic(G_K, metis_part_K, f"METIS K [{NPARTS}]")
except Exception as e:
    print('METIS K failed:', e)

try:
    metis_part_M = partition_graph_metis(G_M, nparts=NPARTS, weight='weight', seed=123)
    summarize_generic(G_M, metis_part_M, f"METIS M [{NPARTS}]")
except Exception as e:
    print('METIS M failed:', e)

METIS K [5]: cut=8391.6201, parts=[0, 1, 2, 3, 4], per=[6706.458127358207, 6733.513206070527, 6729.453954886778, 6727.261356945957, 6749.178328580379], total=33645.8650
METIS M [5]: cut=13125.1420, parts=[0, 1, 2, 3, 4], per=[36247.69678502237, 35888.10202492381, 36593.69668642173, 36448.5468040664, 36044.84030662915], total=181222.8826


## Parameter Tuning (balance_tol, refine_passes, n_trials)

This cell performs a lightweight grid search over a few settings to improve cut quality while respecting balance.
It applies to the multilevel pipeline (bipartition or recursive K-way). If METIS mode is selected, it will skip.


In [9]:
# Skip tuning for direct METIS or direct K-way mode
if PARTITION_MODE in ('kway_metis', 'kway_direct'):
    print('Tuning skipped: direct K-way/METIS mode selected.')
else:
    import itertools
    from copy import deepcopy

    def evaluate(G, params, nparts=2):
        if PARTITION_MODE == 'bipartition' and nparts == 2:
            p = multilevel_bipartition(G, **params)
            c = edge_cut(G, p, weight=params.get('weight', 'weight'))
            return p, c
        elif PARTITION_MODE == 'kway_recursive' and nparts >= 2:
            p = k_way_partition(G, nparts, **params)
            c = edge_cut_kway(G, p, weight=params.get('weight', 'weight'))
            return p, c
        else:
            raise ValueError('Unsupported configuration for tuning')

    def rel_imbalance(G, part, k):
        labels, per, total = kway_balance_info(G, part)
        # Pad missing labels if recursive bisection didn't use all labels densely
        if len(per) < k:
            per = per + [0.0] * (k - len(per))
        target = total / float(k) if k else 0.0
        if target == 0.0:
            return 0.0
        return max(abs(w - target) / target for w in per)

    # Fix balance tolerance for fair comparison
    fixed_tol = 0.03

    # Search spaces (moderately expanded)
    initial_methods = ['COMPONENT_AWARE', 'GGGP']
    refine_methods = ['FM', 'KL']
    rpasses = [9, 11]
    trials = [12, 16]

    # Copy and adjust base params for deeper hierarchies
    baseK = deepcopy(params_K); baseK['balance_tol'] = fixed_tol
    baseM = deepcopy(params_M); baseM['balance_tol'] = fixed_tol

    baseK['max_levels'] = int(baseK.get('max_levels', 20)) + 10
    baseM['max_levels'] = int(baseM.get('max_levels', 20)) + 10
    baseK['coarsen_limit'] = min(int(baseK.get('coarsen_limit', 60)), 40)
    baseM['coarsen_limit'] = min(int(baseM.get('coarsen_limit', 80)), 60)

    bestK = (None, float('inf'), None, None)  # (part, cut, params, imbalance)
    bestM = (None, float('inf'), None, None)

    for im, rm, rp, nt in itertools.product(initial_methods, refine_methods, rpasses, trials):
        candK = deepcopy(baseK); candK.update(initial_method=im, refine_method=rm, refine_passes=rp, n_trials=nt)
        candM = deepcopy(baseM); candM.update(initial_method=im, refine_method=rm, refine_passes=rp, n_trials=nt)

        # Evaluate K; if unbalanced, skip evaluating M to save time
        partK, cutK = evaluate(G_K, candK, nparts=NPARTS)
        imbK = rel_imbalance(G_K, partK, NPARTS)
        if imbK > candK.get('balance_tol', fixed_tol) + 1e-9:
            cutK = float('inf')  # reject unbalanced candidate
        else:
            partM, cutM = evaluate(G_M, candM, nparts=NPARTS)
            imbM = rel_imbalance(G_M, partM, NPARTS)
            if imbM > candM.get('balance_tol', fixed_tol) + 1e-9:
                cutM = float('inf')  # reject unbalanced candidate

            if cutM < bestM[1]:
                bestM = (partM, cutM, deepcopy(candM), imbM)

        if cutK < bestK[1]:
            bestK = (partK, cutK, deepcopy(candK), imbK)

    # Apply best found partitions and params (if any valid were found)
    part_K, cut_K, params_K_best, imb_K = bestK
    part_M, cut_M, params_M_best, imb_M = bestM

    if params_K_best is None or params_M_best is None:
        print('No balanced candidate found within the searched grids (tol=0.03). Consider increasing refine_passes/n_trials further or adjusting parameters more aggressively).')
    else:
        print('Best K:', {
            'cut': cut_K,
            'imbalance': imb_K,
            'params': {k: params_K_best[k] for k in ['initial_method','refine_method','refine_passes','n_trials','balance_tol','max_levels','coarsen_limit']}
        })
        summarize_generic(G_K, part_K, f"K [tuned {PARTITION_MODE} {NPARTS}]")

        print('Best M:', {
            'cut': cut_M,
            'imbalance': imb_M,
            'params': {k: params_M_best[k] for k in ['initial_method','refine_method','refine_passes','n_trials','balance_tol','max_levels','coarsen_limit']}
        })
        summarize_generic(G_M, part_M, f"M [tuned {PARTITION_MODE} {NPARTS}]")

        # Update params variables for downstream cells
        params_K.update(params_K_best)
        params_M.update(params_M_best)


Tuning skipped: direct K-way/METIS mode selected.
